In [3]:
import polars as pl
import json
import os
from procyclingstats import Race, Stage
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl import Workbook,utils
from openpyxl.drawing.image import Image

enlace_o_valor = "race/trofeo-calvia/2026"
#enlace_o_valor="race/gran-premio-miguel-indurain/2026"
nombre_archivo = f"data/{enlace_o_valor.replace('/', '_')}.json"

# Verificar si el archivo ya existe
if not os.path.exists(nombre_archivo):
    print(f"El archivo {nombre_archivo} no existe. Obteniendo datos de la web...")
    race = Race(f"{enlace_o_valor}/overview")
    data = race.parse()
    # Guardar en archivo JSON
    with open(nombre_archivo, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f"El archivo {nombre_archivo} ya existe. Cargando datos...")
   
# esto para carreras de un solo día  
with open(nombre_archivo, 'r', encoding='utf-8') as f:
    data = json.load(f) 
print(data)


file = f"data/HISTORIA_{data['name'].replace(' ', '_')}.xlsx"
libros=Workbook()
hojas=libros.active

# Insertar logo en A1
logo = Image('LOGO.png')
logo.width = 250
logo.height =100
hojas.add_image(logo, 'A1')
hojas.cell(row=3, column=5, value=str(data['uci_tour'])+" "+data['name']+":"+str(data['startdate'])) # type: ignore
hojas['C3'].font = Font(color='FF0000', bold=True,size=25)    
c=hojas.max_row+2
#miramos si existen resultados previos
# Acumular datos de todas las ediciones
data_acumulado = {}
edicion=f"data/Resultados_{enlace_o_valor.strip()[0:len(enlace_o_valor)-4].replace('/', '_')}.json"
if not os.path.exists(edicion):
        print(f"El archivo {edicion} no existe. Obteniendo datos de la web...")
        for res in data['prev_editions_select'][1:6]:
            print(f"  Obteniendo datos de {res['value']}...")
            stage = Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{res['text']}/result")
            resultado = stage.parse()
            # Acumular datos en diccionario
            data_acumulado[res['text']] = resultado
        # Guardar en archivo JSON una sola vez después de acumular
        with open(edicion,'w', encoding='utf-8') as f:
            json.dump(data_acumulado, f, indent=4, ensure_ascii=False)
        print(f"El archivo {edicion} ya existe. Cargando datos...")
    #cargamos resultados


for edit in data['prev_editions_select'][0:6]:
    c=hojas.max_row+2
    year = edit['text']
    print(f"Procesando edición {year}...")
    
    with open(edicion, 'r', encoding='utf-8') as f:
        results = json.load(f) 
    print(f"  ✓ Datos de {year} obtenidos"+str(results.keys()))

    #pintamos resumen en hoja principal
    cell=hojas.cell(row=c+2, column=1, value=f"Procesando edición {year}")
    cell.font = Font(bold=True,size=15)
    #past_edit=Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{edition['text']}/result")
    cell = hojas.cell(row=c+3, column=1, value=f'{results[year]['won_how']}')
    cell.font = Font(color='FF0000',bold=True)
    cell=hojas.cell(row=c+3, column=2, value=f'participacion:{results[year]['race_startlist_quality_score']}' )
    cell.font = Font(bold=True)
    cell=hojas.cell(row=c+3, column=3, value=f'desnivel: {results[year]['vertical_meters']}m')
    cell.font = Font(bold=True)
    cell=hojas.cell(row=c+3, column=4, value=f'distancia {results[year]['distance']}km')
    cell.font = Font(bold=True)
    cell=hojas.cell(row=c+3, column=5, value=f'avg:{results[year]['avg_speed_winner']}km/h') 
    cell.font = Font(bold=True)

    
# Ajustar ancho de columnas al contenido
for column_cells in hojas.columns:
    max_length = 0
    column_letter = utils.get_column_letter(column_cells[0].column)
    for cell in column_cells:
        if cell.value is not None:
           max_length = max(max_length, len(str(cell.value)))
    hojas.column_dimensions[column_letter].width = min(max_length + 2, 60)
libros.save(file)
'''
# Eliminar archivo temporal si existe
if os.path.exists('temp_grafico_especialidades.png'):
        try:
            os.remove('temp_grafico_especialidades.png')
            print(f"✅ Gráfico insertado en celda  (imagen temporal eliminada)")
        except Exception as e:
            print(f"✅ Gráfico insertado en celda  (⚠️ no se pudo eliminar imagen temporal: {e})")
else:
        print(f"✅ Gráfico insertado en celda  (⚠️ archivo temporal no encontrado)")


# Cargar resultados de ediciones anteriores
res_file = f"Resultados_{enlace_o_valor.strip()[0:len(enlace_o_valor)-4].replace('/', '_')}.json"

if not os.path.exists(res_file):
    print(f"✗ El archivo {res_file} NO existe. Obteniendo resultados de la web...")
    # Inicializar diccionario para acumular todas las ediciones
    data_res = {}
    
    # Obtener todas las ediciones anteriores
    for edition in data['prev_editions_select'][1:6]:
        year = edition['text']
        print(f"  Obteniendo datos de {year}...")
        try:
            past_edit = Stage(f"{enlace_o_valor.strip()[0:len(enlace_o_valor)-4]}{year}/result")
            data_res[year] = past_edit.parse()
            print(f"  ✓ Datos de {year} obtenidos")
        except Exception as e:
            print(f"  ✗ Error al obtener datos de {year}: {e}")
    
    # Guardar en archivo JSON
    with open(res_file, 'w', encoding='utf-8') as f:
        json.dump(data_res, f, indent=4, ensure_ascii=False)
    print(f"\nTodas las ediciones guardadas en {res_file}")
'''
print("\n=== DATOS DE LA CARRERA ===")
print(f"Carrera: {data.get('name', 'N/A')}")
print(f"Fecha: {data.get('date', 'N/A')}")
print(f"Distancia: {data.get('distance', 'N/A')} km")
'''
print("\n=== RESULTADOS ===")
print(f"Total de ediciones almacenadas: {len(data_res)}")
for year, resultado in data_res.items():
    if 'results' in resultado and resultado['results']:
        ganador = resultado['results'][0]['rider_name']
        print(f"{year}: {ganador} ({len(resultado['results'])} corredores)")
'''

El archivo data/race_trofeo-calvia_2026.json no existe. Obteniendo datos de la web...
El archivo data/race_trofeo-calvia_2026.json ya existe. Cargando datos...
{'category': 'Men Elite', 'edition': 25, 'enddate': '2026-01-28', 'is_one_day_race': True, 'name': 'Trofeo Calvià', 'nationality': 'ES', 'prev_editions_select': [{'text': '2026', 'value': 'race/trofeo-calvia/2026/statistics/start'}, {'text': '2025', 'value': 'race/trofeo-calvia/2025/statistics/start'}, {'text': '2024', 'value': 'race/trofeo-calvia/2024/statistics/start'}, {'text': '2023', 'value': 'race/trofeo-calvia/2023/statistics/start'}, {'text': '2022', 'value': 'race/trofeo-calvia/2022/statistics/start'}, {'text': '2021', 'value': 'race/trofeo-calvia/2021/statistics/start'}, {'text': '2011', 'value': 'race/trofeo-calvia/2011/statistics/start'}, {'text': '2010', 'value': 'race/trofeo-calvia/2010/statistics/start'}, {'text': '2009', 'value': 'race/trofeo-calvia/2009/statistics/start'}, {'text': '2008', 'value': 'race/trofeo-

KeyError: '2026'

In [20]:
# Cargar datos de resultados
import polars as pl
from procyclingstats import Rider

enlace_o_valor = "race/gran-premio-miguel-indurain/2026"
edicion = f"data/Resultados_{enlace_o_valor.strip()[0:len(enlace_o_valor)-4].replace('/', '_')}.json"

with open(edicion, 'r', encoding='utf-8') as f:
    data_resultados = json.load(f)

df_especialidades_todos: list[pl.DataFrame] = []

# Mostrar los 10 primeros de cada edición
print("=== TOP 10 RESULTADOS POR EDICIÓN ===\n")

for year, resultado in data_resultados.items():
    if 'results' in resultado and resultado['results']:
        especialidades: list[str] = []
        df = pl.DataFrame(resultado['results'], strict=False)
        print("=== TOP 10 RESULTADOS POR EDICIÓN ===\n")
        print(df.head(10))
        df_uci = df.group_by('team_name').agg(pl.sum('uci_points').alias('total_uci_points'))
        print(f"\n📄 EDICIÓN {year}")
        print("Columnas:", df.columns)
        print("Primeras filas:\n", df_uci.sort('total_uci_points', descending=True))
        for idx, txirrindu in enumerate(df.head(10).select('rider_url', 'rider_name').iter_rows()):
            rider_url, rider_name = txirrindu
            try:
                rider = Rider(str(rider_url))  # type: ignore
                points_per_speciality = rider.parse()['points_per_speciality']
                if isinstance(points_per_speciality, dict):
                    sorted_by_values = dict(sorted(points_per_speciality.items(), key=lambda item: item[1], reverse=True))
                    df_rider_specialidad = pl.DataFrame([sorted_by_values])
                else:
                    df_rider_specialidad = pl.DataFrame(points_per_speciality)

                df_rider_specialidad = df_rider_specialidad.with_columns([
                    pl.lit(rider_name).alias('rider_name'),
                    pl.lit(idx + 1).alias('position')
                ])
                df_especialidades_todos.append(df_rider_specialidad)

                cols = list(df_rider_specialidad.columns)
                row_values = df_rider_specialidad.row(0)
                cols_especialidad = [c for c in cols if c not in ['rider_name', 'edition', 'position']]
                if len(cols_especialidad) >= 2:
                    especialidad = (
                        f"{cols_especialidad[0]}:{row_values[cols.index(cols_especialidad[0])]},"
                        f"{cols_especialidad[1]}:{row_values[cols.index(cols_especialidad[1])]}"
                    )
                elif len(cols_especialidad) == 1:
                    especialidad = f"{cols_especialidad[0]}:{row_values[cols.index(cols_especialidad[0])]}"
                else:
                    especialidad = "sin datos"
            except Exception as e:
                print(f"⚠️ Error procesando rider {rider_name}: {str(e)}")
                especialidad = "Error al obtener datos"

            especialidades.append(especialidad)

        if len(especialidades) > 0:
            df_res_como = df.head(len(especialidades)).with_columns(pl.Series('especialidad', especialidades))
        else:
            print(f"\n📄 EDICIÓN {year}: sin resultados")
        df_res_como = df.head(len(especialidades)).with_columns(pl.Series('especialidad', especialidades))
        print(df_res_como)
        resultado['results'] = df_res_como.to_dicts()
    else:
        print(f"\n📄 EDICIÓN {year}: sin resultados")
    with open(edicion, 'w', encoding='utf-8') as f: 
        json.dump(resultado, f, indent=4, ensure_ascii=False)


=== TOP 10 RESULTADOS POR EDICIÓN ===

=== TOP 10 RESULTADOS POR EDICIÓN ===

shape: (10, 14)
┌────────────┬───────────┬───────────┬───────────┬───┬─────────┬───────────┬───────────┬───────────┐
│ rider_name ┆ rider_url ┆ rider_num ┆ team_name ┆ … ┆ bonus   ┆ pcs_point ┆ uci_point ┆ breakaway │
│ ---        ┆ ---       ┆ ber       ┆ ---       ┆   ┆ ---     ┆ s         ┆ s         ┆ _kms      │
│ str        ┆ str       ┆ ---       ┆ str       ┆   ┆ str     ┆ ---       ┆ ---       ┆ ---       │
│            ┆           ┆ i64       ┆           ┆   ┆         ┆ i64       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═════════╪═══════════╪═══════════╪═══════════╡
│ Nys Thibau ┆ rider/thi ┆ 44        ┆ Lidl -    ┆ … ┆ 0:00:00 ┆ 125       ┆ 200.0     ┆ 0.0       │
│            ┆ bau-nys   ┆           ┆ Trek      ┆   ┆         ┆           ┆           ┆           │
│ Molenaar   ┆ rider/ale ┆ 114       ┆ Caja      ┆ … ┆ 0:00:00 ┆ 85        ┆ 150.0     ┆ 0.0      

In [15]:
from filecmp import DEFAULT_IGNORES
import polars as pl
import json

# Convertir resultados a Polars DataFrame
print("\n=== CONVERTIR A POLARS ===\n")
enlace_o_valor = "race/trofeo-calvia/2026"
nombre_archivo = f"data/{enlace_o_valor.replace('/', '_')}.json"

# Cargar resultados de ediciones anteriores
res_file = f"data/Resultados_{enlace_o_valor.strip()[0:len(enlace_o_valor)-4].replace('/', '_')}.json"

df_edit = pl.read_json(nombre_archivo)
print(df_edit)
df_res=pl.read_json(res_file)
print(df_res)

'''
# Leer JSON directamente con Python
with open(nombre_archivo, 'r', encoding='utf-8') as f:
    data_json = json.load(f)
print(f"Datos cargados del archivo {nombre_archivo}")
print(data_json)
# Crear lista con todos los datos normalizados
registros = []

for año, datos in data_json.items():
    # Agregar el año a cada registro
    registro = datos.copy()
   # registro['año'] = año
    registros.append(registro)

# Crear DataFrame normalizado desde la lista con strict=False para tipos mixtos
#df_normalizado = pl.DataFrame(data_json, strict=False)

print("=== DATAFRAME NORMALIZADO ===")
print(f"\nColumnas: {df_normalizado.columns}")


# Si quieres ver con más detalle las primeras filas
print("\n=== PRIMERAS 3 FILAS ===")
print(df_normalizado['won_how','año','avg_speed_winner','vertical_meters'])
print(df_normalizado['results'][0]) '''



=== CONVERTIR A POLARS ===

shape: (1, 12)
┌───────────┬─────────┬────────────┬──────────────┬───┬─────────────┬────────────┬──────────┬──────┐
│ category  ┆ edition ┆ enddate    ┆ is_one_day_r ┆ … ┆ stages_winn ┆ startdate  ┆ uci_tour ┆ year │
│ ---       ┆ ---     ┆ ---        ┆ ace          ┆   ┆ ers         ┆ ---        ┆ ---      ┆ ---  │
│ str       ┆ i64     ┆ str        ┆ ---          ┆   ┆ ---         ┆ str        ┆ str      ┆ i64  │
│           ┆         ┆            ┆ bool         ┆   ┆ list[null]  ┆            ┆          ┆      │
╞═══════════╪═════════╪════════════╪══════════════╪═══╪═════════════╪════════════╪══════════╪══════╡
│ Men Elite ┆ 25      ┆ 2026-01-28 ┆ true         ┆ … ┆ []          ┆ 2026-01-28 ┆ 1.1      ┆ 2026 │
└───────────┴─────────┴────────────┴──────────────┴───┴─────────────┴────────────┴──────────┴──────┘
shape: (1, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ 2025              ┆ 2024       

'\n# Leer JSON directamente con Python\nwith open(nombre_archivo, \'r\', encoding=\'utf-8\') as f:\n    data_json = json.load(f)\nprint(f"Datos cargados del archivo {nombre_archivo}")\nprint(data_json)\n# Crear lista con todos los datos normalizados\nregistros = []\n\nfor año, datos in data_json.items():\n    # Agregar el año a cada registro\n    registro = datos.copy()\n   # registro[\'año\'] = año\n    registros.append(registro)\n\n# Crear DataFrame normalizado desde la lista con strict=False para tipos mixtos\n#df_normalizado = pl.DataFrame(data_json, strict=False)\n\nprint("=== DATAFRAME NORMALIZADO ===")\nprint(f"\nColumnas: {df_normalizado.columns}")\n\n\n# Si quieres ver con más detalle las primeras filas\nprint("\n=== PRIMERAS 3 FILAS ===")\nprint(df_normalizado[\'won_how\',\'año\',\'avg_speed_winner\',\'vertical_meters\'])\nprint(df_normalizado[\'results\'][0]) '